In [ ]:
from ase.io import read, write
from ase.visualize import view
from ase.lattice.cubic import FaceCenteredCubic
import numpy as np
from ase import Atoms

In [ ]:
atoms = read("bulk_contcar", format = "vasp")
latticeconstant = atoms.get_cell()[0,0]
atoms = FaceCenteredCubic("Cu", latticeconstant=latticeconstant)
supercell = atoms*(2,2,2)

In [ ]:
view(atoms)
view(supercell)

In [ ]:
def substitute_homogeneously(atoms, symbol_from='Cu', symbol_to='Ga', n_sub=1):
    positions = atoms.get_positions()
    indices = np.arange(len(atoms))

    # Track which atoms can be replaced
    copper_indices = [i for i, atom in enumerate(atoms) if atom.symbol == symbol_from]
    # print(copper_indices)

    # Pick the first substitution randomly
    chosen = [np.random.choice(copper_indices)]
    copper_indices.remove(chosen[0])

    for _ in range(n_sub - 1):
        # For each candidate, compute min distance to already chosen
        distances = []
        for i in copper_indices:
            dists = np.linalg.norm(positions[i] - positions[chosen], axis=1)
            distances.append(np.min(dists))
        
        # Choose atom that is farthest from all current Ga atoms
        best_index = copper_indices[np.argmax(distances)]
        chosen.append(best_index)
        copper_indices.remove(best_index)

    # Perform substitution
    for idx in chosen:
        atoms[idx].symbol = symbol_to
    
    return atoms



In [ ]:

def substitute_homogeneously(atoms, symbol_from='Cu', symbol_to='Ga', n_sub=1):
    """
    Substitutes `n_sub` atoms of type `symbol_from` with `symbol_to` in a way that spreads
    them out as homogeneously as possible, considering periodic boundary conditions.

    Parameters:
        atoms (ase.Atoms): The structure to modify.
        symbol_from (str): Atomic symbol to replace.
        symbol_to (str): Atomic symbol to insert.
        n_sub (int): Number of atoms to substitute.

    Returns:
        ase.Atoms: A modified copy of the input atoms with substitutions.
    """
    if not isinstance(atoms, Atoms):
        raise TypeError("`atoms` must be an ASE Atoms object.")
    if not isinstance(symbol_from, str) or not isinstance(symbol_to, str):
        raise TypeError("`symbol_from` and `symbol_to` must be strings.")
    if not isinstance(n_sub, int) or n_sub < 1:
        raise ValueError("`n_sub` must be a positive integer.")

    # positions = atoms.get_positions()
    eligible_indices = [i for i, atom in enumerate(atoms) if atom.symbol == symbol_from]

    if n_sub > len(eligible_indices):
        raise ValueError(f"Requested {n_sub} substitutions, but only {len(eligible_indices)} '{symbol_from}' atoms found.")

    chosen = []

    # Choose the first substitution randomly
    first = 0
    # first = np.random.choice(eligible_indices)
    chosen.append(first)
    eligible_indices.remove(first)

    for _ in range(n_sub - 1):
        max_min_dist = -np.inf
        best_index = None

        for idx in eligible_indices:
            dists = atoms.get_distances(idx, chosen, mic=True)
            min_dist = np.min(dists)

            if min_dist > max_min_dist:
                max_min_dist = min_dist
                best_index = idx

        if best_index is not None:
            chosen.append(best_index)
            eligible_indices.remove(best_index)
        else:
            raise RuntimeError("No suitable atom found for substitution. Possibly a logic error.")

    # Make a copy to avoid modifying original
    atoms = atoms.copy()
    for idx in chosen:
        atoms[idx].symbol = symbol_to

    return atoms


def substitute_islands(atoms, symbol_from='Cu', symbol_to='Ga', n_sub=1):
    """
    Substitutes `n_sub` atoms of type `symbol_from` with `symbol_to` in a way that spreads
    them out as homogeneously as possible, considering periodic boundary conditions.

    Parameters:
        atoms (ase.Atoms): The structure to modify.
        symbol_from (str): Atomic symbol to replace.
        symbol_to (str): Atomic symbol to insert.
        n_sub (int): Number of atoms to substitute.

    Returns:
        ase.Atoms: A modified copy of the input atoms with substitutions.
    """
    if not isinstance(atoms, Atoms):
        raise TypeError("`atoms` must be an ASE Atoms object.")
    if not isinstance(symbol_from, str) or not isinstance(symbol_to, str):
        raise TypeError("`symbol_from` and `symbol_to` must be strings.")
    if not isinstance(n_sub, int) or n_sub < 1:
        raise ValueError("`n_sub` must be a positive integer.")

    eligible_indices = [i for i, atom in enumerate(atoms) if atom.symbol == symbol_from]

    if n_sub > len(eligible_indices):
        raise ValueError(f"Requested {n_sub} substitutions, but only {len(eligible_indices)} '{symbol_from}' atoms found.")

    chosen = []

    # Choose the first substitution randomly
    first = 0
    # first = np.random.choice(eligible_indices)
    chosen.append(first)
    eligible_indices.remove(first)

    for _ in range(n_sub - 1):
        max_min_dist = np.inf
        best_index = None

        for idx in eligible_indices:
            dists = atoms.get_distances(idx, chosen, mic=True)
            min_dist = np.min(dists)

            if min_dist < max_min_dist:
                max_min_dist = min_dist
                best_index = idx

        if best_index is not None:
            chosen.append(best_index)
            eligible_indices.remove(best_index)
        else:
            raise RuntimeError("No suitable atom found for substitution. Possibly a logic error.")

    # Make a copy to avoid modifying original
    atoms = atoms.copy()
    for idx in chosen:
        atoms[idx].symbol = symbol_to

    return atoms

In [ ]:
for i in range(1, 9):
    new = substitute_homogeneously(supercell, n_sub=i)
    # view(new)
    write(f"0{i}", new, format = "vasp")

In [ ]:
write("00", supercell, format = "vasp")

In [ ]:
new = substitute_homogeneously(supercell, n_sub=8)

In [ ]:
for i in range(1, 9):
    new = substitute_homogeneously(supercell, n_sub=i)
    write(f"0{i}", new, format = "vasp", sort=True)